# HyperGLOT — Stage A comparison (Google Colab)

This notebook clones the **GLOT repo (your fork, with the Stage A changes)** from git and runs a 4-way comparison of token-graph construction:

| Config | `--graph_metric` | `--graph_adj` | Meaning |
|---|---|---|---|
| **A. GLOT threshold** (original) | `cosine` | `threshold` | edge if `cosine(x_i,x_j) > tau` |
| **B. GLOT kNN** | `cosine` | `knn` | k nearest by cosine |
| **C. HyperGLOT threshold** | `poincare` | `threshold` | edge if `d_Poincare < rho` |
| **D. HyperGLOT kNN** | `poincare` | `knn` | k nearest by hyperbolic distance |

The frozen LLM backbone is unchanged; only the pooling head's graph construction differs.

> **Runtime:** Set `Runtime -> Change runtime type -> GPU` (a free T4 is plenty for BERT). Encoding the frozen backbone happens once and is cached; each config then trains the tiny GLOT head quickly.

## 0. Prerequisite — push your local GLOT folder to GitHub

This notebook reads the code **from git**, so your local `GLOT` folder (containing `hyperbolic_graph.py` and the patched `main.py` / `diagnostic_stress_test.py`) must be on GitHub. From the `GNN/GLOT` folder on your machine:

```bash
cd "GNN/GLOT"
git checkout -b hyperglot-stageA          # optional: work on a branch
git add hyperbolic_graph.py verify_stage_a.py main.py diagnostic_stress_test.py HYPERGLOT_PHASE0-1.md
git commit -m "Stage A: hyperbolic token-graph construction"
# create an empty repo on GitHub, then point 'origin' at it (or add a new remote):
git remote add hyperglot https://github.com/<YOUR_USERNAME>/GLOT.git
git push -u hyperglot hyperglot-stageA
```

Then set `REPO_URL` and `BRANCH` in the config cell below.

In [ ]:
# ================== CONFIG ==================
REPO_URL = "https://github.com/amita1212/GLOT.git"
BRANCH   = "hyperglot-stageA"                       # <-- branch you pushed (or 'main')

HF_TOKEN = ""          # optional: your HuggingFace token (bert-base-uncased is public, so blank is fine)
BACKBONE = "bert-base-uncased"
TASKS    = ["cola", "rte"]   # cheap, relational GLUE tasks. add: sst2, stsb, mrpc, qnli...
EPOCHS   = 3
SEED     = 42

# Graph hyper-parameters (matched across configs where applicable)
TAU       = 0.8    # cosine threshold (paper's GLUE value; README uses --tau=0.8)
RHO       = 2.0    # hyperbolic-distance threshold
CURVATURE = 1.0    # Poincare ball curvature c
KNN_K     = 8      # neighbours for kNN configs

# ---- Checkpointing / debugging ----
# Results are appended to this CSV after EACH (task, config) run, so a crash
# mid-sweep never loses finished runs. Set RESUME=True to skip rows already there.
RESULTS_CSV = "/content/hyperglot_results.csv"
LOGS_DIR    = "/content/logs"          # full stdout+stderr of every run is saved here
RESUME      = True                      # skip (task, config) pairs already in RESULTS_CSV
DEBUG       = True                      # stream live progress + print graph-building stats

# Keep experiment tracking quiet/offline on Colab
import os
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
os.makedirs(LOGS_DIR, exist_ok=True)
print("Config set. Results ->", RESULTS_CSV, "| logs ->", LOGS_DIR)

In [ ]:
# Check GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

In [ ]:
# ================== CLONE REPO ==================
import os, shutil, subprocess
REPO_DIR = "/content/GLOT"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Cloned into", REPO_DIR)
print(sorted(os.listdir(REPO_DIR)))

In [ ]:
# ================== SANITY: Stage A present? ==================
assert os.path.exists("hyperbolic_graph.py"), \
    "hyperbolic_graph.py missing -- did you push your fork with the Stage A changes?"
main_src = open("main.py", encoding="utf-8").read()
assert "graph_metric" in main_src, \
    "main.py has no --graph_metric flag -- push the patched main.py from your fork."
print("Stage A code detected in the cloned repo.")

In [ ]:
# ================== PATCH: GLUE dataset id ==================
# The upstream GLOT loads GLUE via load_dataset("glue", ...), which current
# datasets/huggingface_hub reject (HfUriError: repo id must be 'namespace/name').
# Rewrite the cloned main.py to the canonical parquet mirror "nyu-mll/glue".
# Idempotent: a no-op if your fork already contains the fix.
_mainp = os.path.join(REPO_DIR, "main.py")
_src = open(_mainp, encoding="utf-8").read()
_n = _src.count('load_dataset("glue"')
if _n:
    _src = _src.replace('load_dataset("glue"', 'load_dataset("nyu-mll/glue"')
    open(_mainp, "w", encoding="utf-8").write(_src)
    print(f"Patched {_n} GLUE loader(s) -> nyu-mll/glue")
else:
    print("No 'glue' loader to patch (fork already fixed).")


In [ ]:
# ================== PATCH: stress-test figure crash ==================
# diagnostic_stress_test.py finishes training, then tries to append the pooled
# GLOT vector (dim != backbone hidden when jk_mode=cat) to a token-similarity
# figure, crashing with "Sizes of tensors must match ... 768 vs 1280".
# Only concat the pooled vector when its dim matches. Idempotent no-op if already fixed.
_sp = os.path.join(REPO_DIR, "diagnostic_stress_test.py")
_s = open(_sp, encoding="utf-8").read()
_bad = (
    '    dim = z.size(-1)\n'
    '    pooled_ours = z[0]\n'
    '    all_labels.append("[Token-GNN]")\n'
    '    print(z.shape)\n'
    '    all_vectors = torch.cat([all_vectors.detach().cpu(), z[0].unsqueeze(0).detach().cpu()], dim=0).cpu().numpy()\n'
)
_good = (
    '    print(z.shape)\n'
    '    if z.size(-1) == all_vectors.size(-1):\n'
    '        all_labels.append("[Token-GNN]")\n'
    '        all_vectors = torch.cat([all_vectors.detach().cpu(), z[0].unsqueeze(0).detach().cpu()], dim=0).cpu().numpy()\n'
    '    else:\n'
    '        print(f"[stress] pooled dim {z.size(-1)} != token dim {all_vectors.size(-1)}; omitting pooled vector from figure.")\n'
    '        all_vectors = all_vectors.detach().cpu().numpy()\n'
)
if _bad in _s:
    open(_sp, "w", encoding="utf-8").write(_s.replace(_bad, _good))
    print("Patched diagnostic_stress_test.py figure crash.")
else:
    print("No stress-figure crash to patch (fork already fixed).")


In [ ]:
# ================== INSTALL DEPENDENCIES ==================
# Colab ships torch+CUDA. Install PyG companion wheels matching the installed torch, then the rest.
import torch, sys, subprocess
v  = torch.__version__.split("+")[0]
cu = ("cu" + torch.version.cuda.replace(".", "")) if torch.cuda.is_available() and torch.version.cuda else "cpu"
wheel_index = f"https://data.pyg.org/whl/torch-{v}+{cu}.html"
print("PyG wheel index:", wheel_index)

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args])

# Compiled PyG extras (best effort). If the exact wheel is unavailable, the shim cell below covers torch_scatter.
pip("torch-scatter", "torch-sparse", "torch-cluster", "-f", wheel_index)
pip("torch-geometric", "geoopt", "transformers>=4.40", "datasets", "mteb", "ranx", "peft", "wandb", "scikit-learn")
print("Dependencies installed.")

In [ ]:
# ================== FALLBACK: torch_scatter shim ==================
# main.py only uses torch_scatter.scatter_add. If the compiled wheel didn't install
# for this exact torch build, drop a tiny pure-PyTorch shim on the path so imports work.
try:
    import torch_scatter  # noqa
    print("torch_scatter available (compiled).")
except Exception:
    shim = '''
import torch
def scatter_add(src, index, dim=-1, out=None, dim_size=None):
    if dim < 0: dim = src.dim() + dim
    if dim_size is None:
        dim_size = int(index.max().item()) + 1 if index.numel() else 0
    shape = list(src.shape); shape[dim] = dim_size
    target = src.new_zeros(shape) if out is None else out
    idx = index
    if idx.dim() == 1 and src.dim() > 1:
        view = [1] * src.dim(); view[dim] = -1
        idx = idx.view(view).expand_as(src)
    return target.scatter_add_(dim, idx, src)
'''
    with open(os.path.join(REPO_DIR, "torch_scatter.py"), "w", encoding="utf-8") as f:
        f.write(shim)
    print("Installed pure-PyTorch torch_scatter shim.")

## 1a. Graph-building diagnostics (what each config does)

Before training, inspect **exactly how each config builds the token graph** on real BERT embeddings. This shows, per config, the number of edges, average degree, isolated nodes, and edge-weight range — so you can *see* the difference between cosine/poincaré and threshold/kNN, and catch a mis-tuned `rho`/`tau` (e.g. **0 edges** = threshold too tight) before spending time on a full training run.


In [ ]:
# ================== GRAPH-BUILDING DIAGNOSTICS ==================
# Build each config's token graph on a few real sentences and report stats.
import sys, torch
import pandas as pd
sys.path.insert(0, REPO_DIR)
from transformers import AutoTokenizer, AutoModel
from hyperbolic_graph import HyperbolicGraphConfig, build_pyg_graphs_hyper

_dev = "cuda" if torch.cuda.is_available() else "cpu"
_tok = AutoTokenizer.from_pretrained(BACKBONE)
_enc = AutoModel.from_pretrained(BACKBONE).to(_dev).eval()

_demo = [
    "The cat sat on the mat while the dog slept nearby.",
    "Although it was raining heavily, they decided to go for a long walk.",
]
_batch = _tok(_demo, padding=True, truncation=True, max_length=64, return_tensors="pt").to(_dev)
with torch.no_grad():
    _hidden = _enc(**_batch).last_hidden_state          # (B, L, d)
_mask = _batch["attention_mask"]
print(f"hidden={tuple(_hidden.shape)}  tokens/seq={_mask.sum(1).tolist()}  device={_dev}\n")

# Same four configs as the training sweep, expressed as graph builders.
CFG_MAP = {
    "A_glot_threshold":  HyperbolicGraphConfig(graph_metric="cosine",   adjacency="threshold", tau=TAU),
    "B_glot_knn":        HyperbolicGraphConfig(graph_metric="cosine",   adjacency="knn",       k=KNN_K),
    "C_hyper_threshold": HyperbolicGraphConfig(graph_metric="poincare", adjacency="threshold", rho=RHO,   curvature=CURVATURE),
    "D_hyper_knn":       HyperbolicGraphConfig(graph_metric="poincare", adjacency="knn",       k=KNN_K,   curvature=CURVATURE),
}

def _graph_stats(cfg):
    g = build_pyg_graphs_hyper(_hidden, _mask, cfg, device=_hidden.device)
    n = int(g.num_nodes)
    E = int(g.edge_index.size(1))
    deg = torch.bincount(g.edge_index[0], minlength=n).float() if E else torch.zeros(n)
    w = g.edge_attr.view(-1)
    return {
        "nodes": n,
        "edges": E,
        "avg_degree": round(E / max(n, 1), 2),
        "max_degree": int(deg.max().item()) if n else 0,
        "isolated_nodes": int((deg == 0).sum().item()),
        "edge_w_min": round(float(w.min()), 3) if w.numel() else None,
        "edge_w_max": round(float(w.max()), 3) if w.numel() else None,
    }

_rows = []
for _name, _cfg in CFG_MAP.items():
    _s = _graph_stats(_cfg)
    _rows.append({"config": _name, **_s})
    _flag = ""
    if _s["edges"] == 0:
        _flag = "  <-- WARNING: 0 edges (threshold too tight; lower rho / tau)"
    elif _s["isolated_nodes"] > 0:
        _flag = f"  <-- note: {_s['isolated_nodes']} isolated node(s)"
    print(f"[{_name}] {_s}{_flag}")

graph_diag = pd.DataFrame(_rows)
graph_diag.to_csv("/content/graph_diagnostics.csv", index=False)
print("\nSaved graph diagnostics -> /content/graph_diagnostics.csv")
display(graph_diag)


## 1. Run the 4-way comparison on GLUE

Each config trains only the GLOT pooling head on top of the **frozen** BERT backbone. `--precompute_hidden_states=1` caches the backbone's token states so the sweep is fast.

In [ ]:
import re, subprocess, sys, os, datetime
import pandas as pd

# The four graph-construction configurations (all use pooling_method=glot).
CONFIGS = {
    "A_glot_threshold":  ["--graph_metric=cosine",   "--graph_adj=threshold", f"--tau={TAU}"],
    "B_glot_knn":        ["--graph_metric=cosine",   "--graph_adj=knn",       f"--knn_k={KNN_K}"],
    "C_hyper_threshold": ["--graph_metric=poincare", "--graph_adj=threshold", f"--rho={RHO}", f"--curvature={CURVATURE}"],
    "D_hyper_knn":       ["--graph_metric=poincare", "--graph_adj=knn",       f"--knn_k={KNN_K}", f"--curvature={CURVATURE}"],
}

RESULT_COLS = ["task", "config", "final_metric_line", "status", "log_file", "timestamp"]

def _load_done():
    """Return the set of (task, config) already recorded, for resume."""
    if RESUME and os.path.exists(RESULTS_CSV):
        df = pd.read_csv(RESULTS_CSV)
        return set(zip(df["task"].astype(str), df["config"].astype(str)))
    return set()

def _append_row(row):
    """Append one result row to the CSV immediately (crash-safe checkpoint)."""
    write_header = not os.path.exists(RESULTS_CSV)
    pd.DataFrame([row], columns=RESULT_COLS).to_csv(RESULTS_CSV, mode="a", header=write_header, index=False)

# Lines worth echoing live so you can watch each stage (encode, graph, train).
_LIVE = re.compile(r"epoch|graph|edge|node|precompute|encod|train|eval|nan|error|traceback|"
                   r"acc|f1|spearman|pearson|mcc|loss", re.I)

def stream_run(cmd, log_path, tag):
    """Run cmd, stream key lines live to the cell AND write the full log to disk."""
    print("\n" + "=" * 80)
    print(tag)
    print("  cmd:", " ".join(cmd))
    print("  log:", log_path)
    print("=" * 80, flush=True)
    lines = []
    with open(log_path, "w", encoding="utf-8") as lf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            lines.append(line)
            lf.write(line)
            lf.flush()
            if DEBUG and _LIVE.search(line):
                print("  " + line.rstrip(), flush=True)
        proc.wait()
    return "".join(lines), proc.returncode

def run_config(task, name, extra):
    cmd = [
        sys.executable, "main.py",
        f"--model_name_or_path={BACKBONE}",
        f"--task={task}",
        "--pooling_method=glot", "--gnn_type=gat",
        "--num_layers=2", "--jk_mode=cat",
        "--gat_hidden_dim=256", "--scorer_hidden=128", "--proj_dim=256",
        "--max_length=128", f"--epochs={EPOCHS}", "--batch_size=32",
        "--eval_batch_size=64", "--lr=2e-4", "--weight_decay=0.0",
        f"--seed={SEED}", "--verbose=1",
        "--precompute_hidden_states=1", "--override_precompute=0",
        "--finetune_backbone=0", "--decoder_cls_last_token=0", "--adaptive_length=0",
    ] + extra
    log_path = os.path.join(LOGS_DIR, f"{task}__{name}.log")
    out, rc = stream_run(cmd, log_path, f"[{task}] {name}: {' '.join(extra)}")
    metric_lines = [ln for ln in out.splitlines() if re.search(r"\[.*\]\s*epoch\s*\d+", ln)]
    final = metric_lines[-1].strip() if metric_lines else "(no metric line parsed -- see log)"
    status = "ok" if rc == 0 else f"FAILED(rc={rc})"
    return final, status, log_path

done = _load_done()
print(f"Resume: {len(done)} run(s) already in {RESULTS_CSV}" if done else "Starting fresh.")

for task in TASKS:
    for name, extra in CONFIGS.items():
        if (task, name) in done:
            print(f"[skip] {task} / {name}  (already done)")
            continue
        final, status, log_path = run_config(task, name, extra)
        _append_row({
            "task": task, "config": name, "final_metric_line": final,
            "status": status, "log_file": log_path,
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        })
        print(f"  -> [{status}] {final}   (saved to {RESULTS_CSV})")

results = pd.read_csv(RESULTS_CSV)
print("\n\nDONE. All results so far:")


In [ ]:
# ================== RESULTS TABLE ==================
# Reads the checkpoint CSV (written incrementally during the sweep), so this
# shows whatever has finished even if the run above crashed partway.
import pandas as pd
pd.set_option("display.max_colwidth", None)
results = pd.read_csv(RESULTS_CSV)
display(results)
print(f"{len(results)} run(s) in {RESULTS_CSV}")
failed = results[results["status"] != "ok"]
if len(failed):
    print(f"\n{len(failed)} failed run(s) -- inspect their logs:")
    for _, r in failed.iterrows():
        print(f"  {r['task']}/{r['config']}: {r['log_file']}")


## 2. (Optional) Negation stress test

The signal-in-noise diagnostic (`diagnostic_stress_test.py`) — where GLOT keeps >97% accuracy at 90% distractors. Run the same 4 configs and compare robustness. Increase `--distractor_ratio` toward 0.9 for the hard regime.

In [ ]:
STRESS_DISTRACTOR = 0.8
STRESS_CSV = "/content/hyperglot_stress.csv"
STRESS_COLS = ["config", "distractor_ratio", "result", "status", "log_file", "timestamp"]

def _stress_done():
    if RESUME and os.path.exists(STRESS_CSV):
        df = pd.read_csv(STRESS_CSV)
        return set(zip(df["config"].astype(str), df["distractor_ratio"].astype(float)))
    return set()

def run_stress(name, extra):
    cmd = [
        sys.executable, "diagnostic_stress_test.py",
        f"--model_name_or_path={BACKBONE}",
        "--pooling_method=glot", "--num_layers=2", "--jk_mode=cat",
        "--gat_hidden_dim=256", "--scorer_hidden=128",
        "--max_length=128", f"--distractor_ratio={STRESS_DISTRACTOR}",
        f"--seed={SEED}", "--verbose=1",
    ] + extra
    log_path = os.path.join(LOGS_DIR, f"stress_{name}_d{STRESS_DISTRACTOR}.log")
    out, rc = stream_run(cmd, log_path, f"STRESS {name}: {' '.join(extra)}")
    # The headline number is "Best Validation Accuracy"; fall back to any acc line.
    best = [ln for ln in out.splitlines() if "Best Validation Accuracy" in ln]
    acc_lines = best or [ln for ln in out.splitlines() if re.search(r"acc|accuracy", ln, re.I)]
    result = acc_lines[-1].strip() if acc_lines else "(see log)"
    # A non-zero exit only from the optional post-training figure step still yields
    # a valid accuracy; mark ok if we captured the Best Validation Accuracy.
    status = "ok" if (rc == 0 or best) else f"FAILED(rc={rc})"
    return result, status, log_path

done_s = _stress_done()
for name, extra in CONFIGS.items():
    if (name, float(STRESS_DISTRACTOR)) in done_s:
        print(f"[skip] stress {name}  (already done)")
        continue
    result, status, log_path = run_stress(name, extra)
    row = {"config": name, "distractor_ratio": STRESS_DISTRACTOR, "result": result,
           "status": status, "log_file": log_path,
           "timestamp": datetime.datetime.now().isoformat(timespec="seconds")}
    write_header = not os.path.exists(STRESS_CSV)
    pd.DataFrame([row], columns=STRESS_COLS).to_csv(STRESS_CSV, mode="a", header=write_header, index=False)
    print(f"  -> [{status}] {result}   (saved to {STRESS_CSV})")

stress = pd.read_csv(STRESS_CSV)
display(stress)